# CSR synaptic-current reordering benchmark

This notebook tests whether physically sorting recurrent synapses by delayed presynaptic id is a useful direction for `calculate_synaptic_currents`.

It compares the current ragged-indirection implementation against `models.calculate_synaptic_current_optimized`, which expects CSR-ordered arrays and uses a chunked fused edge traversal for `d(rec_z_buf)`. The optimized function is a TensorFlow reference path for deciding whether a CUDA/TensorFlow op is worth building.

In [1]:
from pathlib import Path
import pickle as pkl
import sys
import time

import numpy as np
import tensorflow as tf

REPO = Path.cwd()
if not (REPO / 'v1_model_utils').exists():
    REPO = Path('/home/jgalvan/Desktop/Neurocoding/V1_GLIF_model')
sys.path.insert(0, str(REPO))

from v1_model_utils import models

print('repo:', REPO)
print('tensorflow:', tf.__version__)
print('gpus:', tf.config.list_physical_devices('GPU'))

2026-06-20 18:21:04.278155: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-20 18:21:04.278220: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-20 18:21:04.279050: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-20 18:21:04.285076: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-06-20 18:21:05.226566: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


repo: /home/jgalvan/Desktop/Neurocoding/V1_GLIF_model
tensorflow: 2.15.0
gpus: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:4', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:5', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:6', device_type='GPU')]


In [2]:
# Default: small cached full-network sample, quick enough for CPU sanity checks.
# For a more realistic GPU benchmark, change CACHE_PATH to a larger cached model
# or build arrays from your run's loaded `network` object.
CACHE_PATH = REPO / 'GLIF_network_nll_full/tf_data/V1_network_v1_1000.pkl'
BASIS_PATH = REPO / 'GLIF_network_nll_full/tf_data/syn_id_to_syn_weights_dict.pkl'

MAX_DELAY = 3
BATCH_SIZE = 3
SPIKE_SPARSITY = 0.04
DTYPE = tf.float16
GRAD_CHUNK_SIZE = 1 << 16
BENCH_REPEATS = 5

assert CACHE_PATH.exists(), CACHE_PATH
assert BASIS_PATH.exists(), BASIS_PATH

In [3]:
def load_cached_network(cache_path):
    with open(cache_path, 'rb') as f:
        network, lgn_input, bkg_input = pkl.load(f)
    return network


def load_basis(path, dtype=np.float32):
    with open(path, 'rb') as f:
        return np.asarray(list(pkl.load(f).values()), dtype=dtype)


def delayed_recurrent_arrays(network, max_delay):
    syn = network['synapses']
    indices = np.asarray(syn['indices']).copy()
    weights = np.asarray(syn['weights'], dtype=np.float32).copy()
    syn_ids = np.asarray(syn['syn_ids'], dtype=np.int64).copy()
    delays = np.round(np.clip(np.asarray(syn['delays'], dtype=np.float32), 1, max_delay)).astype(np.int32)
    n_neurons = int(network['n_nodes'])
    indices[:, 1] = indices[:, 1] + n_neurons * (delays - 1)
    dense_shape = (int(syn['dense_shape'][0]), max_delay * int(syn['dense_shape'][1]))
    return indices, weights, syn_ids, dense_shape


def build_current_layout(indices, weights, syn_ids, dense_shape):
    pre_ind_table = models.make_pre_ind_table(
        indices,
        n_source_neurons=dense_shape[1],
        order_keys=(indices[:, 0], syn_ids),
    )
    return {
        'pre_ind_table': pre_ind_table,
        'indices': tf.constant(indices.astype(np.int64)),
        'post_ids': tf.constant(indices[:, 0].astype(np.int64)),
        'weights_np': weights,
        'syn_ids': tf.constant(syn_ids.astype(np.int64)),
    }


def build_csr_layout(indices, weights, syn_ids, dense_shape):
    pre_ind_table = models.make_pre_ind_table(
        indices,
        n_source_neurons=dense_shape[1],
        order_keys=(indices[:, 0], syn_ids),
    )
    order = pre_ind_table.flat_values.numpy().astype(np.int32)
    csr_indices = indices[order]
    return {
        'order': order,
        'row_splits': pre_ind_table.row_splits,
        'post_ids': tf.constant(csr_indices[:, 0].astype(np.int32)),
        'pre_ids': tf.constant(csr_indices[:, 1].astype(np.int32)),
        'weights_np': weights[order],
        'syn_ids': tf.constant(syn_ids[order].astype(np.int64)),
    }


def random_spike_buffer(batch_size, n_pre, sparsity, dtype, seed=1):
    rng = np.random.default_rng(seed)
    rec = (rng.random((batch_size, n_pre)) < sparsity).astype(np.float32)
    return tf.Variable(tf.cast(rec, dtype))


def bytes_gb(nbytes):
    return nbytes / 1024**3

In [4]:
network = load_cached_network(CACHE_PATH)
basis_np = load_basis(BASIS_PATH)
indices, weights, syn_ids, dense_shape = delayed_recurrent_arrays(network, MAX_DELAY)
dense_shape_tf = (
    tf.constant(dense_shape[0], dtype=tf.int64),
    tf.constant(dense_shape[1], dtype=tf.int64),
)
current = build_current_layout(indices, weights, syn_ids, dense_shape)
csr = build_csr_layout(indices, weights, syn_ids, dense_shape)
basis = tf.constant(tf.cast(basis_np, DTYPE))

n_edges = len(weights)
n_post, n_pre = dense_shape
row_lengths = np.diff(csr['row_splits'].numpy())

print('n_nodes:', network['n_nodes'])
print('dense_shape:', dense_shape)
print('n_edges:', n_edges)
print('receptors:', basis_np.shape[1])
print('row length mean/p95/max:', float(row_lengths.mean()), int(np.percentile(row_lengths, 95)), int(row_lengths.max()))
print('empty delayed-pre rows:', int(np.sum(row_lengths == 0)))

current_static_gb = bytes_gb(indices.astype(np.int64).nbytes + current['pre_ind_table'].flat_values.numpy().nbytes)
csr_static_gb = bytes_gb(
    csr['post_ids'].numpy().nbytes
    + csr['pre_ids'].numpy().nbytes
    + csr['syn_ids'].numpy().nbytes
    + csr['row_splits'].numpy().nbytes
)
print('approx current static index GB:', current_static_gb)
print('approx csr static index GB:', csr_static_gb)

n_nodes: 1000
dense_shape: (1000, 3000)
n_edges: 36817
receptors: 4
row length mean/p95/max: 12.272333333333334 26 309
empty delayed-pre rows: 900
approx current static index GB: 0.0006857700645923615
approx csr static index GB: 0.0005597956478595734


2026-06-20 18:21:07.989884: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 859 MB memory:  -> device: 0, name: NVIDIA RTX 6000 Ada Generation, pci bus id: 0000:01:00.0, compute capability: 8.9
2026-06-20 18:21:07.991685: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22059 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:23:00.0, compute capability: 8.6
2026-06-20 18:21:07.993229: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 22059 MB memory:  -> device: 2, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:41:00.0, compute capability: 8.6
2026-06-20 18:21:07.994674: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 22059 MB memory:  -> device: 3, name: NVIDIA GeForce RTX

In [5]:
rec = random_spike_buffer(BATCH_SIZE, n_pre, SPIKE_SPARSITY, DTYPE)
w_current = tf.Variable(tf.cast(current['weights_np'], DTYPE))
w_csr = tf.Variable(tf.cast(csr['weights_np'], DTYPE))

with tf.GradientTape(persistent=True) as tape:
    y_current = models.calculate_synaptic_currents(
        rec,
        current['indices'],
        current['post_ids'],
        w_current,
        w_current,
        dense_shape_tf,
        basis,
        current['syn_ids'],
        current['pre_ind_table'],
    )
    y_csr = models.calculate_synaptic_current_optimized(
        rec,
        csr['post_ids'],
        csr['pre_ids'],
        w_csr,
        w_csr,
        csr['row_splits'],
        dense_shape_tf,
        basis,
        csr['syn_ids'],
        grad_chunk_size=GRAD_CHUNK_SIZE,
    )
    probe = tf.reshape(tf.linspace(tf.cast(0.1, DTYPE), tf.cast(1.0, DTYPE), tf.size(y_current)), tf.shape(y_current))
    loss_current = tf.reduce_sum(y_current * probe)
    loss_csr = tf.reduce_sum(y_csr * probe)

drec_current = tape.gradient(loss_current, rec)
drec_csr = tape.gradient(loss_csr, rec)
dw_current = tape.gradient(loss_current, w_current)
dw_csr = tape.gradient(loss_csr, w_csr)
dw_csr_original = np.empty_like(current['weights_np'])
dw_csr_original[csr['order']] = dw_csr.numpy()

drec_diff = tf.cast(drec_current - drec_csr, tf.float32)
drec_ref = tf.cast(drec_current, tf.float32)
drec_rel_l2 = tf.linalg.norm(drec_diff) / tf.maximum(tf.linalg.norm(drec_ref), tf.constant(1e-12, tf.float32))
print('forward max abs:', float(tf.reduce_max(tf.abs(y_current - y_csr)).numpy()))
print('drec max abs:', float(tf.reduce_max(tf.abs(drec_current - drec_csr)).numpy()))
print('drec ref max abs:', float(tf.reduce_max(tf.abs(drec_current)).numpy()))
print('drec relative l2:', float(drec_rel_l2.numpy()))
print('dw max abs:', float(np.max(np.abs(dw_current.numpy() - dw_csr_original))))

forward max abs: 0.25
drec max abs: 2.0
drec ref max abs: 3700.0
drec relative l2: 0.000402375211706385
dw max abs: 0.001953125


In [6]:
@tf.function
def current_forward_backward(rec, weights_var, probe):
    with tf.GradientTape(persistent=True) as tape:
        y = models.calculate_synaptic_currents(
            rec,
            current['indices'],
            current['post_ids'],
            weights_var,
            weights_var,
            dense_shape_tf,
            basis,
            current['syn_ids'],
            current['pre_ind_table'],
        )
        loss = tf.reduce_sum(y * probe)
    drec = tape.gradient(loss, rec)
    dw = tape.gradient(loss, weights_var)
    return loss, tf.reduce_sum(drec), tf.reduce_sum(dw)


@tf.function
def csr_forward_backward(rec, weights_var, probe):
    with tf.GradientTape(persistent=True) as tape:
        y = models.calculate_synaptic_current_optimized(
            rec,
            csr['post_ids'],
            csr['pre_ids'],
            weights_var,
            weights_var,
            csr['row_splits'],
            dense_shape_tf,
            basis,
            csr['syn_ids'],
            grad_chunk_size=GRAD_CHUNK_SIZE,
        )
        loss = tf.reduce_sum(y * probe)
    drec = tape.gradient(loss, rec)
    dw = tape.gradient(loss, weights_var)
    return loss, tf.reduce_sum(drec), tf.reduce_sum(dw)


def time_call(label, fn, *args, repeats=BENCH_REPEATS):
    fn(*args)
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        out = fn(*args)
        _ = [x.numpy() for x in out]
        times.append(time.perf_counter() - t0)
    print(label, 'median_s=', float(np.median(times)), 'all_s=', [round(t, 5) for t in times])
    return np.asarray(times)


probe = tf.ones((BATCH_SIZE * n_post, basis_np.shape[1]), dtype=DTYPE)
times_current = time_call('current', current_forward_backward, rec, w_current, probe)
times_csr = time_call('csr_optimized', csr_forward_backward, rec, w_csr, probe)
print('median csr/current:', float(np.median(times_csr) / np.median(times_current)))

current median_s= 0.0026154560036957264 all_s= [0.00387, 0.00268, 0.00254, 0.00257, 0.00262]


csr_optimized median_s= 0.00247546611353755 all_s= [0.00286, 0.00221, 0.00248, 0.00252, 0.00193]
median csr/current: 0.9464759147313638


## How to interpret

- If `csr_optimized` is faster or close on GPU, physical CSR ordering plus a fused CUDA op is very likely worth it.
- If this TensorFlow prototype is slower but the static index memory estimate is much lower, CSR ordering can still be worthwhile as a memory layout change before a custom op.
- If `drec max abs` is small and `dw max abs` is zero or near zero, differences are accumulation-order effects rather than a changed objective.